# Actividad 5. Técnicas de visualización de resultados en Big Data
**Actividad 5**

Jesús Santiago Orduño Bennett - 01797766**

En esta actividad se retoma el algoritmo con mejor desempeño del Proyecto de la etapa 3 RandomForest sobre clasificación binaria de usuarios `member_casual` y aplica un proceso de validación cruzada estratificada para medir la variabilidad de los resultados, visualizando el desempeño por pliegue y la detección de sobreajuste.

**Tarea de aprendizaje:** clasificación binaria (member vs casual).
**Métricas:** AUC-ROC (primaria), F1-Score, Accuracy, Precision/Recall por clase.

---
- Declaración de uso de herramientas de IA
La responsabilidad final sobre el contenido recae en el autor.
Anthropic. (2026). Claude (Opus 4.x) [Modelo de lenguaje grande], utilizado para consultaría, corrección de errores, y revisión de redacción. https://claude.ai/

In [15]:
!pip -q install pyspark
from pyspark.sql import SparkSession, functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd

spark = SparkSession.builder.appName("Actividad5_Equipo41").getOrCreate()

SEED = 41
DATA_PATH = "/content/Aggregated_Data.csv"
# variable objetivo
TARGET_COL = "member_casual"
SAMPLE_FRACTION = 0.80

P = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
P = P.dropDuplicates().na.drop(how="all")
P = P.na.drop(subset=[TARGET_COL])
n_P = P.count()

# Muestra M: muestreo ESTRATIFICADO por la variable objetivo
strata = [r[TARGET_COL] for r in P.select(TARGET_COL).distinct().collect()]
fractions = {v: SAMPLE_FRACTION for v in strata}
M = P.stat.sampleBy(TARGET_COL, fractions, seed=SEED).cache()
n_M = M.count()

print(f"Registros en P: {n_P:,}")
print(f"Registros en M: {n_M:,}  ({round(n_M/n_P*100,2)}% de P)")
print(f"Variable objetivo: {TARGET_COL}")
M.groupBy(TARGET_COL).count().show()

Registros en P: 6,004,093
Registros en M: 4,802,574  (79.99% de P)
Variable objetivo: member_casual
+-------------+-------+
|member_casual|  count|
+-------------+-------+
|       casual|1968985|
|       member|2833589|
+-------------+-------+



## 1. Cálculo del valor K para validación cruzada

Partiendo de la muestra representativa M (derivada de las variables de caracterización de P mediante muestreo estratificado), se determina el número de pliegues **k = 5**.

**Justificación:**
- **Sesgo-varianza:** un k muy pequeño (2–3) entrena cada modelo con muy pocos
  datos y sesga la estimación; un k cercano a LOOCV dispara la varianza del
  estimador y el costo computacional. El rango 5–10 es el equilibrio establecido
  en la literatura (Purushotham & Tripathy, 2011, *stratified tenfold CV*).
- **Representatividad de cada fold:** al ser M un volumen grande, incluso con k=5
  cada pliegue conserva decenas de miles de registros. Con estratificación por
  `member_casual`, cada fold preserva la proporción de clases (~60/40); por la ley
  de los grandes números, sus distribuciones convergen a las de la población P.
  La representatividad no es el factor limitante: lo es el costo.
- **Costo en Big Data:** k=10 implica el doble de entrenamientos que k=5 con una
  mejora marginal del estimador. La actividad advierte que más pliegues encarecen
  la experimentación, por lo que k=5 es el compromiso adecuado: estimación de bajo
  sesgo y varianza aceptable, a la mitad del costo. (k=10 sería válido con más cómputo.)

In [16]:
K = 5

dist_M = (M.groupBy(TARGET_COL).count()
            .withColumn("pct", F.round(F.col("count")/F.lit(n_M)*100, 2)))
print("Distribución de clases en M (se preservará en cada fold):")
dist_M.show()


print("Tamaño aproximado por pliegue según k:")
for k in [3, 5, 10]:
    val = n_M // k
    print(f"  k={k:>2}: validación ≈ {val:,} | entrenamiento ≈ {n_M - val:,}")

Distribución de clases en M (se preservará en cada fold):
+-------------+-------+----+
|member_casual|  count| pct|
+-------------+-------+----+
|       casual|1968985|41.0|
|       member|2833589|59.0|
+-------------+-------+----+

Tamaño aproximado por pliegue según k:
  k= 3: validación ≈ 1,600,858 | entrenamiento ≈ 3,201,716
  k= 5: validación ≈ 960,514 | entrenamiento ≈ 3,842,060
  k=10: validación ≈ 480,257 | entrenamiento ≈ 4,322,317


## 2. Construcción de los k-folds

Se construyen los k=5 pliegues mediante **asignación estratificada**: dentro de
cada clase de `member_casual`, las filas se ordenan de forma aleatoria reproducible
y se reparten cíclicamente entre los 5 folds. Así cada pliegue queda balanceado en
clases (misma proporción que M y, por tanto, que P), lo que garantiza que cada fold
sea una muestra representativa de la población. Se valida explícitamente la
proporción de clases por fold.

In [17]:
from pyspark.sql import Window

M_folds = M.withColumn("_rand", F.rand(seed=SEED))
w_estrato = Window.partitionBy(TARGET_COL).orderBy("_rand")
M_folds = (M_folds
           .withColumn("_rn", F.row_number().over(w_estrato))
           .withColumn("fold", F.col("_rn") % F.lit(K))
           .drop("_rand", "_rn")
           .cache())

print("Tamaño de cada fold:")
M_folds.groupBy("fold").count().orderBy("fold").show()

w_fold = Window.partitionBy("fold")
print("Proporción de clases por fold (debe conservar el ~60/40 de M):")
(M_folds.groupBy("fold", TARGET_COL).count()
        .withColumn("pct_en_fold",
                    F.round(F.col("count")/F.sum("count").over(w_fold)*100, 2))
        .orderBy("fold", TARGET_COL)
        .show(50, truncate=False))

Tamaño de cada fold:
+----+------+
|fold| count|
+----+------+
|   0|960514|
|   1|960515|
|   2|960515|
|   3|960515|
|   4|960515|
+----+------+

Proporción de clases por fold (debe conservar el ~60/40 de M):
+----+-------------+------+-----------+
|fold|member_casual|count |pct_en_fold|
+----+-------------+------+-----------+
|0   |casual       |393797|41.0       |
|0   |member       |566717|59.0       |
|1   |casual       |393797|41.0       |
|1   |member       |566718|59.0       |
|2   |casual       |393797|41.0       |
|2   |member       |566718|59.0       |
|3   |casual       |393797|41.0       |
|3   |member       |566718|59.0       |
|4   |casual       |393797|41.0       |
|4   |member       |566718|59.0       |
+----+-------------+------+-----------+



## 3. Experimentación con Validación Cruzada

Se entrena el algoritmo con mejor desempeño de la Etapa 3 (**RandomForest**) sobre los k=5 pliegues. En cada iteración un fold actúa como validación y los 4 restantes como entrenamiento.

**Selección de variables:** se descartan el identificador `ride_id`, las marcas de tiempo crudas y las columnas de estación (alta cardinalidad, que inflarían el vector sin mejorar la generalización). Se conservan como numéricas la duración (`ride_length`, `hms_lapse`), la geolocalización (`start/end_lat/lng`) y la hora de inicio derivada (`start_hour`); como categóricas, `rideable_type` y `day_of_week` (One-Hot).

La etiqueta es binaria con member = clase positiva

**Detección de sobreajuste:** en cada fold se mide el AUC tanto en entrenamiento
como en validación; una brecha grande indicaría sobreajuste.

**Métricas (Etapa 3):** AUC-ROC (primaria), F1, Accuracy y Precision/Recall por clase.

In [18]:
# Etiqueta binaria explícita: member = clase positiva (1)
data = M_folds.withColumn("label", (F.col(TARGET_COL) == "member").cast("double"))
data = data.withColumn("start_hour", F.hour("started_at"))   # feature derivada

cat_cols = ["rideable_type", "day_of_week"]
num_cols = ["start_lat", "start_lng", "end_lat", "end_lng",
            "ride_length", "hms_lapse", "start_hour"]

data = data.na.drop(subset=cat_cols + num_cols + ["label"]).cache()


In [23]:
# --- Pipeline ---
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
            for c in cat_cols]
encoder  = OneHotEncoder(inputCols=[c+"_idx" for c in cat_cols],
                         outputCols=[c+"_ohe" for c in cat_cols])
assembler = VectorAssembler(inputCols=num_cols + [c+"_ohe" for c in cat_cols],
                            outputCol="features")
rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                            numTrees=30, maxDepth=5, seed=SEED)
pipeline = Pipeline(stages=indexers + [encoder, assembler, rf])

In [24]:
# --- Evaluadores ---
ev_auc  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                        metricName="areaUnderROC")
ev_f1   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
ev_acc  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
ev_p1   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="precisionByLabel", metricLabel=1.0)
ev_r1   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="recallByLabel", metricLabel=1.0)
ev_p0   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="precisionByLabel", metricLabel=0.0)
ev_r0   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="recallByLabel", metricLabel=0.0)


In [25]:
# Validación cruzada estratificada
resultados = []
pred_para_viz = None

for i in range(K):
    train_df = data.filter(F.col("fold") != i)
    val_df   = data.filter(F.col("fold") == i)

    model = pipeline.fit(train_df)
    pred_train = model.transform(train_df)
    pred_val   = model.transform(val_df)

    fila = {
        "fold": i,
        "AUC_train": ev_auc.evaluate(pred_train),
        "AUC_val":   ev_auc.evaluate(pred_val),
        "F1_val":    ev_f1.evaluate(pred_val),
        "Accuracy_val": ev_acc.evaluate(pred_val),
        "Precision_member": ev_p1.evaluate(pred_val),
        "Recall_member":    ev_r1.evaluate(pred_val),
        "Precision_casual": ev_p0.evaluate(pred_val),
        "Recall_casual":    ev_r0.evaluate(pred_val),
    }
    resultados.append(fila)
    print(f"Fold {i}: AUC_val={fila['AUC_val']:.4f} | F1={fila['F1_val']:.4f} | "
          f"Acc={fila['Accuracy_val']:.4f} | gap_AUC={fila['AUC_train']-fila['AUC_val']:.4f}")

    if i == 0:
        pred_para_viz = pred_val.cache()   # para ROC y matriz de confusión

df_resultados = pd.DataFrame(resultados)
print("\nResumen por fold:")
print(df_resultados.round(4).to_string(index=False))

print("\nVariabilidad entre folds (media ± desv. estándar):")
print(df_resultados.drop(columns=["fold"]).agg(["mean", "std"]).round(4).to_string())

Fold 0: AUC_val=0.6951 | F1=0.6137 | Acc=0.6604 | gap_AUC=0.0010
Fold 1: AUC_val=0.6952 | F1=0.6156 | Acc=0.6602 | gap_AUC=0.0009
Fold 2: AUC_val=0.6972 | F1=0.6143 | Acc=0.6615 | gap_AUC=-0.0010
Fold 3: AUC_val=0.6974 | F1=0.6147 | Acc=0.6614 | gap_AUC=-0.0011
Fold 4: AUC_val=0.6957 | F1=0.6106 | Acc=0.6583 | gap_AUC=0.0008

Resumen por fold:
 fold  AUC_train  AUC_val  F1_val  Accuracy_val  Precision_member  Recall_member  Precision_casual  Recall_casual
    0     0.6961   0.6951  0.6137        0.6604            0.6483         0.9283            0.7261         0.2739
    1     0.6960   0.6952  0.6156        0.6602            0.6493         0.9229            0.7168         0.2814
    2     0.6962   0.6972  0.6143        0.6615            0.6487         0.9309            0.7326         0.2730
    3     0.6963   0.6974  0.6147        0.6614            0.6488         0.9297            0.7303         0.2744
    4     0.6965   0.6957  0.6106        0.6583            0.6467         0.9283    